# Phase 1.10 — the extreme-tile encoder cache (tile 32UQC, 346 cubes)

**This notebook computes nothing scientific.** It builds three artefacts so the
extreme-tile P3 can be fitted locally on CPU:

| artefact | files | what it is |
|---|---|---|
| `embeddings/` | 346 × 5 = **1730** `.npz` | the RGB cache, `TIER_A` |
| `embeddings_cir/` | 346 × 4 = **1384** `.npz` | the colour-infrared cache, `TIER_A_CIR` |
| `masks/` | **346** `.npz` | per-pixel clear masks, needed for common-masking |

**Why this runs on Colab and the fitting does not.** Encoding is the only
GPU-bound work in the project, and it is also the only step that needs Python
≥ 3.10: `dinov2_vitb14` loads its code from `torch.hub` and that code uses PEP
604 unions (`X | None`), which are a syntax error on 3.9. The project's dev venv
is 3.9.6, so **two of the nine views — `dinov2_vitb14` and `dinov2_vitb14_cir` —
cannot be built there at all.** Colab is 3.11+ and has a T4. Everything
downstream of this notebook is CPU-only by design and runs on the laptop, where
a 12-hour fit is not at the mercy of a session timeout.

**The tile is the `extreme` split, not `train`.** 32UNU (Phases 1.7/1.9) is a
`train` tile; 32UQC is not. Every earlier encoding notebook hardcodes
`download(TILE, "train", ...)`, and copying that here downloads the wrong cubes.

**The roster is 346, not 348, and it is decided elsewhere.** The 2026-08-17 P4
pilot excluded two cubes whose "clear" pixels carry exactly-zero reflectance in
B04 and B8A across 61 grid cells — a no-data fill block the published
`s2_dlmask`/SCL conjunction does not flag. P3 must run on the **same** 346 cubes
or the two tables are not about the same place.


## Step 1: Install, then restart

In [1]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_10_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run against your own environment (pip install -r requirements.txt) "
          "and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    # satlaspretrain-models is REQUIRED here and was not in Phase 1.5's list:
    # that phase read no embeddings, this one builds them.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy satlaspretrain-models

    # Colab ships a CUDA-matched torch. Installing over it swaps in a CPU wheel
    # and makes every encoder far slower, so only act if something is MISSING.
    # This is the one notebook in the project where that distinction costs real
    # wall-clock: everything downstream is CPU-only by design.
    if (importlib.util.find_spec("torch") is None
            or importlib.util.find_spec("torchvision") is None):
        !pip install torch torchvision

    # Verify before restarting, so a broken install cannot reach the encoders.
    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "torch, torchvision, satlaspretrain_models, sklearn, scipy")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the encoders."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 74.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

## Step 2: Bootstrap

The resolver block is character-identical to every phase notebook from 1.3 on and is pinned by `tests/test_notebook_resolver.py`.

In [1]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "encoders/base.py", "encoders/frames.py",
            "encoders/raw_features.py", "encoders/imagenet_vit.py",
            "encoders/dinov2_vit.py", "encoders/satlas_s2.py",
            "encoders/satlas_s2_mi.py", "scripts/scale_p4.py",
            "probes/p3_forecast.py", "probes/p3_triggers.py",
            "probes/p4_ceiling.py",
            "probes/cv.py", "probes/p1_appearance.py", "probes/p2_deltas.py",
            "tests/test_cv_folds.py", "tests/test_p2_deltas.py",
            "tests/conftest.py"]
ZIP_NAME = "phase1_10_repo.zip"
PHASE = "phase1_10"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p3_forecast.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.9 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_10
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_10/ removes
        everything Phase 1.9 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz)")
print("        ^ the 20-cube Phase 1.2 cache. READ-ONLY here, and NEVER written")
print("          to: it is keyed to exactly 20 cubes and every published result")
print("          must stay reproducible from it. Step 10 reads it to prove the")
print("          scaled cache reproduces it on the cubes the two share.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders import TIER_A
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"encoders to cache ({len(TIER_A)}): {TIER_A}")
print("this notebook writes a CACHE, not a result. No probe runs here.")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

Mounted at /content/drive
found /content/drive/MyDrive/NeurIPS-CCAI-2026/phase1_10/phase1_10_repo.zip
extracting into /content/drive/MyDrive/NeurIPS-CCAI-2026/phase1_10 (zip is newer)

THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.
Colab is still showing the cells it opened. To pick up the
new ones: File > Open notebook > Google Drive, and open
   /content/drive/MyDrive/NeurIPS-CCAI-2026/phase1_10/notebooks
Until you do, the .py files are new and these cells are old.

REPO    /content/drive/MyDrive/NeurIPS-CCAI-2026/phase1_10
RAW     /content/drive/MyDrive/NeurIPS-CCAI-2026/data/raw   (115 cubes)
EMB_IN  /content/drive/MyDrive/NeurIPS-CCAI-2026/phase1_2/data/phase1_2/embeddings   (100 .npz)
        ^ the 20-cube Phase 1.2 cache. READ-ONLY here, and NEVER written
          to: it is keyed to exactly 20 cubes and every published result
          must stay reproducible from it. Step 10 reads it to prove the
          scaled cache reproduces it on the cubes the two share.
RESULTS data/phase1

## Step 3: Environment — this phase wants a GPU, and REQUIRES python >= 3.10

Encoding is the only GPU-bound work in the project and the only step that cannot run on the 3.9.6 dev venv. Everything downstream is CPU-only by design and runs on the laptop.

In [2]:
import glob, os, textwrap

import numpy as np, pandas as pd, sklearn, scipy
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__}")

import torch
print(f"torch {torch.__version__}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    DEVICE = "cuda"
    print(f"GPU  {p.name}, {p.total_memory / 1e9:.1f} GB  <- encoding will use this")
else:
    DEVICE = "cpu"
    print(textwrap.dedent("""
        No GPU. This still WORKS -- nothing here needs CUDA -- but 6990 frames
        through nine views will take a while. On Colab: Runtime > Change
        runtime type > T4 GPU, then re-run from Step 2.
    """).strip())
print(f"\nDEVICE = {DEVICE!r}")

# Python >= 3.10, and this one is not cosmetic. dinov2_vitb14 loads its code
# from torch.hub, and that code uses PEP 604 unions (`X | None`), so on 3.9 it
# dies with "unsupported operand type(s) for |: 'type' and 'NoneType'" -- a
# TypeError from inside a downloaded file, five steps after the real cause.
# Measured on this project's own 3.9 venv: the other four encoders build fine
# and only DINOv2 fails, which is exactly the kind of partial failure that
# produces a cache with a hole in it.
import sys
_py = sys.version_info
print(f"python {_py.major}.{_py.minor}.{_py.micro}")
assert _py >= (3, 10), (
    f"python {_py.major}.{_py.minor} is too old for dinov2_vitb14: its "
    "torch.hub code uses `X | None`, which is a syntax error before 3.10. The "
    "other four encoders would build and the cache would silently be missing "
    "one encoder for every cube. Colab is on 3.11+; if you are running this "
    "locally, use a 3.10+ interpreter."
)
print("\nNOTHING here is fine-tuned. Every encoder is frozen: .eval() and")
print("torch.no_grad(), re-asserted on every call by encoders.base.FrozenEncoder.")

numpy 2.0.2 | pandas 2.2.2 | sklearn 1.6.1 | scipy 1.16.3
torch 2.11.0+cu128
GPU  Tesla T4, 15.6 GB  <- encoding will use this

DEVICE = 'cuda'
python 3.12.13

NOTHING here is fine-tuned. Every encoder is frozen: .eval() and
torch.no_grad(), re-asserted on every call by encoders.base.FrozenEncoder.


## Step 4: The 346 cubes, and the three cache directories

The two excluded cube names are **hardcoded below**, not read from
`data/scaled_32UQC/p4_extreme_results.csv`. That CSV is an untracked build
artefact, so `make_zip.sh` (which derives its file list from `git ls-files`)
does not ship it — on Colab there is nothing to read it from. The names travel
in the code and the count is asserted, so a silently different roster fails
here rather than being discovered when P3 and P4 disagree.


In [3]:
TILE, N_CUBES, SPLIT = "32UQC", 348, "extreme"

# The two cubes the 2026-08-17 P4 pilot excluded, verbatim from the
# `cubes_excluded_fill` column of data/scaled_32UQC/p4_extreme_results.csv.
# HARDCODED because that CSV is untracked and does not travel in the zip.
EXCLUDED_FILL = (
    "32UQC_2018-01-28_2018-11-23_1337_1465_441_569_20_100_6_86.nc",
    "32UQC_2018-01-28_2018-11-23_441_569_441_569_6_86_6_86.nc",
)
EXCLUDED_REASON = (
    "every 'clear' pixel in 61 grid cells carries exactly-zero reflectance in "
    "B04 and B8A (no-data fill the s2_dlmask/SCL conjunction does not flag, and "
    "finite_valid_mask cannot demote because the bands are finite)"
)

_scaled_rel = os.path.join("data", f"scaled_{TILE}")
PROJECT_ROOT = os.path.dirname(REPO) if IS_PHASE_CHECKOUT else REPO

CUBES, _cube_cands = _resolve(os.path.join(_scaled_rel, "raw"), "*.nc", "CUBES")
if not glob.glob(os.path.join(CUBES, "*.nc")):
    CUBES = os.path.join(PROJECT_ROOT, _scaled_rel, "raw")
    print(f"[extreme] nothing on disk yet; will download into {CUBES}")

SCALED_ROOT = os.path.dirname(CUBES)
OUT_RGB = os.path.join(SCALED_ROOT, "embeddings")       # TIER_A, 5 views
OUT_CIR = os.path.join(SCALED_ROOT, "embeddings_cir")   # TIER_A_CIR, 4 views
OUT_MSK = os.path.join(SCALED_ROOT, "masks")
for d in (OUT_RGB, OUT_CIR, OUT_MSK):
    os.makedirs(d, exist_ok=True)

# SPLIT is "extreme", NOT "train". 32UNU is a train tile and every earlier
# encoding notebook hardcodes that; 32UQC is not, and the wrong split either
# lists nothing or lists another tile's cubes.
from scripts.scale_p4 import download
ALL_PATHS = download(TILE, SPLIT, N_CUBES, CUBES)
print(f"\n[extreme] {len(ALL_PATHS)} cubes downloaded/on disk for {TILE}/{SPLIT}")

_all_names = {os.path.basename(p) for p in ALL_PATHS}
_missing = [c for c in EXCLUDED_FILL if c not in _all_names]
assert not _missing, (
    f"the excluded cubes are not in the download: {_missing}. Either the split "
    f"or the non-overlap selection differs from the P4 pilot's, and the P3/P4 "
    "comparison would not be about the same place.")

CUBE_PATHS = [p for p in sorted(ALL_PATHS)
              if os.path.basename(p) not in set(EXCLUDED_FILL)]
n = len(CUBE_PATHS)
print(f"[extreme] excluded {len(EXCLUDED_FILL)} cubes: {EXCLUDED_REASON}")
for c in EXCLUDED_FILL:
    print(f"            - {c}")
assert n == 346, (
    f"roster is {n} cubes, not 346. P3 must run on the same set as the P4 "
    "pilot; a silently different roster makes the two tables incomparable.")
print(f"[extreme] ROSTER = {n} cubes\n")

print(f"CUBES    {CUBES}   ({n} of {len(ALL_PATHS)} used)")
print(f"OUT_RGB  {OUT_RGB}   (this notebook writes here: {n} x 5 = {n*5})")
print(f"OUT_CIR  {OUT_CIR}   (this notebook writes here: {n} x 4 = {n*4})")
print(f"OUT_MSK  {OUT_MSK}   (this notebook writes here: {n})")


[extreme] nothing on disk yet; will download into /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/raw
[dl] 32UQC/extreme: 1212 listed, 348 non-overlapping selected (asked 348)
[dl] 348 cubes on disk (348 new), 2214 MB, 174s -> /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/raw

[extreme] 348 cubes downloaded/on disk for 32UQC/extreme
[extreme] excluded 2 cubes: every 'clear' pixel in 61 grid cells carries exactly-zero reflectance in B04 and B8A (no-data fill the s2_dlmask/SCL conjunction does not flag, and finite_valid_mask cannot demote because the bands are finite)
            - 32UQC_2018-01-28_2018-11-23_1337_1465_441_569_20_100_6_86.nc
            - 32UQC_2018-01-28_2018-11-23_441_569_441_569_6_86_6_86.nc
[extreme] ROSTER = 346 cubes

CUBES    /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/raw   (346 of 348 used)
OUT_RGB  /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings   (this notebook writes here: 346 x 5 = 1730)
OUT_CIR  /

## Step 5: Unit tests

The invariant is **0 failed**, never a particular pass count.

In [4]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")


$ /usr/bin/python3 -m pytest tests
........................................................................ [ 12%]
.........................ssss............s.............................. [ 25%]
........................................................................ [ 38%]
.................................s...................................... [ 51%]
..........................sss........................................... [ 64%]
...........ssssssssss................................................... [ 77%]
....ss..............sssss............................................... [ 90%]
.....................................................                    [100%]
531 passed, 26 skipped in 52.28s
[exit 0] /usr/bin/python3 -m pytest tests


## Step 6: Build all nine views

Five RGB (`TIER_A`) and four colour-infrared (`TIER_A_CIR`). `raw_features` has
no `_cir` twin — it already reads all four bands. This is where the two Satlas
weight downloads happen on a fresh runtime.

`build_encoder` strips the `_cir` suffix to find the wrapper and then re-asserts
it against the instance's own `name`, so a variant can never be cached under the
base name.


In [5]:
import time
from encoders import TIER_A, TIER_A_CIR, build_encoder

print(f"TIER_A     ({len(TIER_A)}) = {list(TIER_A)}")
print(f"TIER_A_CIR ({len(TIER_A_CIR)}) = {list(TIER_A_CIR)}")
assert len(TIER_A) + len(TIER_A_CIR) == 9, "this run is nine views"
print("\nSame weights, same extraction recipe, same frame selection in both")
print("caches. The ONLY difference is which three bands reach the stem:")
print("(B8A, B04, B03) for _cir instead of (B04, B03, B02) for rgb.\n")

t0 = time.time()
ENC_RGB = {n_: build_encoder(n_, device=DEVICE, verbose=True) for n_ in TIER_A}
ENC_CIR = {n_: build_encoder(n_, device=DEVICE, verbose=True) for n_ in TIER_A_CIR}
print(f"\n{len(ENC_RGB) + len(ENC_CIR)} encoders ready on {DEVICE} in "
      f"{time.time() - t0:.0f}s")
for name, enc in {**ENC_RGB, **ENC_CIR}.items():
    print(f"  {name:<28} D={enc.embed_dim:<5} grid_D={enc.grid_dim:<5} "
          f"window_len={enc.window_len}")
assert set(ENC_RGB) == set(TIER_A) and set(ENC_CIR) == set(TIER_A_CIR)


TIER_A     (5) = ['raw_features', 'imagenet_vit_b16', 'dinov2_vitb14', 'satlas_s2_swinb_rgb', 'satlas_s2_swinb_mi_rgb']
TIER_A_CIR (4) = ['imagenet_vit_b16_cir', 'dinov2_vitb14_cir', 'satlas_s2_swinb_rgb_cir', 'satlas_s2_swinb_mi_rgb_cir']

Same weights, same extraction recipe, same frame selection in both
caches. The ONLY difference is which three bands reach the stem:
(B8A, B04, B03) for _cir instead of (B04, B03, B02) for rgb.

[raw_features] not a network: no parameters, nothing to freeze
[raw_features] D=35 (probe default) | grid 4x4 x 35
[raw_features] feature recipe: not a network; pooled = 35 whole-frame statistics; grid = the same 35 statistics recomputed independently per 4x4 cell (35 x 16 = 560), NDVI via data.ndvi.ndvi per cell
[raw_features] preprocessing (inside this wrapper, in this order; nothing else is applied):
[raw_features]   1. band stats over ALL pixels of the UNMODIFIED frame (clouds included, same input the network encoders see), NaN-aware so no-data pixels are

100%|██████████| 330M/330M [00:02<00:00, 163MB/s]


[imagenet_vit_b16] frozen: eval()=True, requires_grad=False on all 85.8M params, device=cuda
[imagenet_vit_b16] D=1536 (probe default) | grid 4x4 x 768
[imagenet_vit_b16] feature recipe: last block, post-LayerNorm; pooled = concat(cls_last, patch_mean_last) = 1536; grid = 14x14 patch tokens adaptive-avg-pooled to 4x4
[imagenet_vit_b16] variants: cls_last=768, patch_mean_last=768
[imagenet_vit_b16] preprocessing (inside this wrapper, in this order; nothing else is applied):
[imagenet_vit_b16]   1. RGB from S2 bands: (B04, B03, B02) -> (R, G, B), indices resolved from S2_BANDS by name
[imagenet_vit_b16]   2. non-finite (no-data) pixels -> NONFINITE_FILL, counted and reported; not inpainting, and done before the resize so a NaN cannot smear
[imagenet_vit_b16]   3. antialiased bilinear resize H x W -> 224 x 224 (torchvision ViT-B/16 accepts exactly 224)
[imagenet_vit_b16]   4. normalise with ImageNet mean=(0.485, 0.456, 0.406) std=(0.229, 0.224, 0.225)
[imagenet_vit_b16]   5. extraction: l

/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 241MB/s]


[dinov2_vitb14] frozen: eval()=True, requires_grad=False on all 86.6M params, device=cuda
[dinov2_vitb14] D=3840 (probe default) | grid 4x4 x 768
[dinov2_vitb14] feature recipe: DINOv2 published linear-probe protocol (Oquab et al., TMLR 2023): pooled = concat(cls_last4_concat, patch_mean_last) = 3840; grid = 16x16 last-block patch tokens adaptive-avg-pooled to 4x4
[dinov2_vitb14] variants: cls_last=768, cls_last4_concat=3072, patch_mean_last=768
[dinov2_vitb14] preprocessing (inside this wrapper, in this order; nothing else is applied):
[dinov2_vitb14]   1. RGB from S2 bands: (B04, B03, B02) -> (R, G, B), indices resolved from S2_BANDS by name
[dinov2_vitb14]   2. non-finite (no-data) pixels -> NONFINITE_FILL, counted and reported; not inpainting, and done before the resize so a NaN cannot smear
[dinov2_vitb14]   3. antialiased bilinear resize H x W -> 224 x 224 (224 = 16 x 14, the ViT-B/14 patch size)
[dinov2_vitb14]   4. normalise with ImageNet mean=(0.485, 0.456, 0.406) std=(0.229, 

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


[dinov2_vitb14_cir] frozen: eval()=True, requires_grad=False on all 86.6M params, device=cuda
[dinov2_vitb14_cir] D=3840 (probe default) | grid 4x4 x 768
[dinov2_vitb14_cir] feature recipe: DINOv2 published linear-probe protocol (Oquab et al., TMLR 2023): pooled = concat(cls_last4_concat, patch_mean_last) = 3840; grid = 16x16 last-block patch tokens adaptive-avg-pooled to 4x4
[dinov2_vitb14_cir] variants: cls_last=768, cls_last4_concat=3072, patch_mean_last=768
[dinov2_vitb14_cir] preprocessing (inside this wrapper, in this order; nothing else is applied):
[dinov2_vitb14_cir]   1. RGB from S2 bands: (B04, B03, B02) -> (R, G, B), indices resolved from S2_BANDS by name
[dinov2_vitb14_cir]   2. non-finite (no-data) pixels -> NONFINITE_FILL, counted and reported; not inpainting, and done before the resize so a NaN cannot smear
[dinov2_vitb14_cir]   3. antialiased bilinear resize H x W -> 224 x 224 (224 = 16 x 14, the ViT-B/14 patch size)
[dinov2_vitb14_cir]   4. normalise with ImageNet mea

## Step 7: Smoke test on ONE cube before committing to 3114 files

A shape or dtype error found here costs seconds; found in Step 8 it costs the
run. This also asserts all nine views agree on *which* frames were retained —
they must, because frame selection happens before any network sees anything,
and it is the invariant the whole `_cir`-vs-`_rgb` comparison rests on.


In [6]:
from data.loader import load_cube
from encoders.pipeline import encode_cube

probe = load_cube(CUBE_PATHS[0], verbose=False)
print(f"smoke cube: {os.path.basename(probe.path)}  values {probe.values.shape}\n")

smoke, t0 = {}, time.time()
for name, enc in {**ENC_RGB, **ENC_CIR}.items():
    ec = encode_cube(probe, enc, verbose=False)
    smoke[name] = ec
    assert ec.embeddings.ndim == 2 and ec.embeddings.shape[1] == enc.embed_dim
    assert ec.grid.shape[1:] == (16, enc.grid_dim), ec.grid.shape
    assert np.isfinite(ec.embeddings).all() and np.isfinite(ec.grid).all()
    print(f"  {name:<28} pooled {str(ec.embeddings.shape):<14} "
          f"grid {str(ec.grid.shape):<18} wsd max {ec.window_span_days.max():.0f} d")
dt = time.time() - t0

ts0 = smoke["raw_features"].timestamps
assert all(np.array_equal(ec.timestamps, ts0) for ec in smoke.values()), (
    "the nine views disagree about which frames were retained; frame selection "
    "happens BEFORE any network sees the data, so they cannot legitimately "
    "differ -- and a _cir/_rgb paired difference over different frames is not "
    "a paired difference")
print(f"\nall nine agree on {len(ts0)} retained frames | {dt:.0f}s for "
      f"1 cube x 9 views on {DEVICE}")
print(f"projected for {len(CUBE_PATHS)} cubes: ~{dt * len(CUBE_PATHS) / 60:.0f} min "
      f"({dt * len(CUBE_PATHS) / 3600:.1f} h)")
print("\nThat projection is LINEAR and encoding genuinely is linear in frames "
      "(one forward pass each) -- unlike the loco fold count downstream.")


[loader] dropping 242/300 timesteps with no acquisition
smoke cube: 32UQC_2018-01-28_2018-11-23_1081_1209_1081_1209_16_96_16_96.nc  values (58, 4, 128, 128)

  raw_features                 pooled (19, 35)       grid (19, 16, 35)       wsd max 0 d
  imagenet_vit_b16             pooled (19, 1536)     grid (19, 16, 768)      wsd max 0 d
  dinov2_vitb14                pooled (19, 3840)     grid (19, 16, 768)      wsd max 0 d
  satlas_s2_swinb_rgb          pooled (19, 1024)     grid (19, 16, 1024)     wsd max 0 d
  satlas_s2_swinb_mi_rgb       pooled (19, 1024)     grid (19, 16, 1024)     wsd max 135 d
  imagenet_vit_b16_cir         pooled (19, 1536)     grid (19, 16, 768)      wsd max 0 d
  dinov2_vitb14_cir            pooled (19, 3840)     grid (19, 16, 768)      wsd max 0 d
  satlas_s2_swinb_rgb_cir      pooled (19, 1024)     grid (19, 16, 1024)     wsd max 0 d
  satlas_s2_swinb_mi_rgb_cir   pooled (19, 1024)     grid (19, 16, 1024)     wsd max 135 d

all nine agree on 19 retained frames

## Step 8: The encode — resumable, and a failed cube is dropped WHOLE

3114 `.npz` (1730 RGB + 1384 CIR) plus 346 masks. The loop **skips whatever
already exists** and re-validates it on load rather than trusting the filename,
so a disconnected Colab session is resumed by re-running this cell.

**The rule that matters here:** if any one of a cube's nine views fails, that
cube is removed from *both* caches and from the masks — never left partial,
never filled. A cube present in the RGB cache and absent from the CIR cache
silently changes which rows a paired `_cir`/`_rgb` difference is computed over,
and that paired difference is the one comparison this whole run exists to make.
Nothing downstream can detect it, so it is enforced here.

Heartbeat every 25 cubes, matching `probes.p4_ceiling.build_p4_data`'s
`CUBE_HEARTBEAT_EVERY`.


In [7]:
from encoders.pipeline import (SCHEMA_VERSION, cube_masks, encode_cube,
                               load_encoded, save_encoded, save_masks)

CUBE_HEARTBEAT_EVERY = 25

def _paths_for(stem):
    """Every file this notebook would write for one cube."""
    return ([os.path.join(OUT_RGB, f"{stem}__{n}.npz") for n in TIER_A]
            + [os.path.join(OUT_CIR, f"{stem}__{n}.npz") for n in TIER_A_CIR]
            + [os.path.join(OUT_MSK, f"{stem}__masks.npz")])

def _drop_cube_whole(stem, why):
    """Rule 3: out of BOTH caches and the masks, or not at all."""
    removed = 0
    for p in _paths_for(stem):
        if os.path.exists(p):
            os.remove(p); removed += 1
    print(f"    DROPPED {stem} from both caches + masks ({removed} files): {why}",
          flush=True)

print(f"cache schema v{SCHEMA_VERSION}. An older-schema file is REFUSED on "
      "load, never silently reused.")
print(f"rgb -> {OUT_RGB}")
print(f"cir -> {OUT_CIR}")
print(f"msk -> {OUT_MSK}\n", flush=True)

rows, failures, dropped = [], [], []
t_start = time.time()
for i, path in enumerate(sorted(CUBE_PATHS), 1):
    cube = os.path.basename(path)
    stem = os.path.splitext(cube)[0]

    if all(os.path.exists(p) for p in _paths_for(stem)):
        rows += [{"cube": cube, "encoder": n, "status": "cached"}
                 for n in tuple(TIER_A) + tuple(TIER_A_CIR)]
    else:
        try:
            s = load_cube(path, verbose=False)
        except Exception as e:                  # noqa: BLE001 -- reported, not hidden
            failures.append((cube, "load_cube", f"{type(e).__name__}: {e}"))
            dropped.append((cube, f"load_cube: {type(e).__name__}"))
            _drop_cube_whole(stem, f"load_cube raised {type(e).__name__}")
            continue

        mask_path = os.path.join(OUT_MSK, f"{stem}__masks.npz")
        cube_rows, failed_view = [], None
        try:
            if not os.path.exists(mask_path):
                save_masks(OUT_MSK, cube_masks(s, verbose=False), verbose=False)
            for out_dir, roster, enc_map in ((OUT_RGB, TIER_A, ENC_RGB),
                                             (OUT_CIR, TIER_A_CIR, ENC_CIR)):
                for name in roster:
                    out = os.path.join(out_dir, f"{stem}__{name}.npz")
                    if os.path.exists(out):
                        ec, status = load_encoded(out), "cached"
                    else:
                        ec = encode_cube(s, enc_map[name], verbose=False)
                        save_encoded(out_dir, ec, verbose=False)
                        status = "encoded"
                    cube_rows.append({"cube": cube, "encoder": name,
                                      "status": status,
                                      "T_kept": int(ec.embeddings.shape[0]),
                                      "D": int(ec.embeddings.shape[1])})
        except Exception as e:                  # noqa: BLE001
            failed_view = f"{type(e).__name__}: {e}"

        if failed_view is not None:
            failures.append((cube, "encode", failed_view))
            dropped.append((cube, failed_view))
            _drop_cube_whole(stem, failed_view)
            continue
        rows += cube_rows

    if i % CUBE_HEARTBEAT_EVERY == 0 or i == len(CUBE_PATHS):
        el = time.time() - t_start
        rate = i / el if el > 0 else 0.0
        eta = (len(CUBE_PATHS) - i) / rate if rate > 0 else float("nan")
        print(f"[{i:>3}/{len(CUBE_PATHS)}] elapsed {el/60:6.1f} min | "
              f"{rate:5.3f} cubes/s | ETA {eta/60:6.1f} min | "
              f"{len(dropped)} dropped", flush=True)

import pandas as pd
ENCODE_LOG = pd.DataFrame(rows)
KEPT = sorted({os.path.basename(p) for p in CUBE_PATHS}
              - {c for c, _ in dropped})
print(f"\nencoded/cached {len(ENCODE_LOG)} (cube, view) pairs in "
      f"{(time.time() - t_start)/60:.1f} min")
print(f"cubes kept {len(KEPT)} | dropped {len(dropped)}")
for c, why in dropped:
    print(f"  DROPPED {c}: {why}")
if dropped:
    print("\nThese are out of BOTH caches and the masks. The local P3 run reads "
          "cache_roster.csv (Step 11) and fits exactly this set.")


cache schema v3. An older-schema file is REFUSED on load, never silently reused.
rgb -> /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings
cir -> /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings_cir
msk -> /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/masks

[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] dropping 242/300 timesteps with no acquisition
[loader] droppin

## Step 9: Audit both caches — every kept cube × every view, or it is unusable

A cube silently missing one view turns every per-encoder comparison into a
comparison over *different cubes*, and no downstream assertion can detect that.
This is `encoders.pipeline.audit_embeddings` / `assert_embeddings_complete`,
pointed at each cache in turn.


In [8]:
from encoders.pipeline import (assert_embeddings_complete, audit_embeddings,
                               print_embedding_audit)

CUBE_IDS = set(KEPT)
print(f"auditing {len(CUBE_IDS)} kept cubes\n")

AUDIT_RGB = audit_embeddings(OUT_RGB, cube_ids=CUBE_IDS)
assert_embeddings_complete(AUDIT_RGB, CUBE_IDS, TIER_A)
print()
AUDIT_CIR = audit_embeddings(OUT_CIR, cube_ids=CUBE_IDS)
assert_embeddings_complete(AUDIT_CIR, CUBE_IDS, TIER_A_CIR)

n_rgb = len(glob.glob(os.path.join(OUT_RGB, "*.npz")))
n_cir = len(glob.glob(os.path.join(OUT_CIR, "*.npz")))
n_msk = len(glob.glob(os.path.join(OUT_MSK, "*.npz")))
print(f"\nrgb embeddings {n_rgb} .npz  (expected {len(CUBE_IDS)} x {len(TIER_A)} "
      f"= {len(CUBE_IDS)*len(TIER_A)})")
print(f"cir embeddings {n_cir} .npz  (expected {len(CUBE_IDS)} x "
      f"{len(TIER_A_CIR)} = {len(CUBE_IDS)*len(TIER_A_CIR)})")
print(f"masks          {n_msk} .npz  (expected {len(CUBE_IDS)})")
assert n_rgb == len(CUBE_IDS) * len(TIER_A), "the rgb cache has holes"
assert n_cir == len(CUBE_IDS) * len(TIER_A_CIR), "the cir cache has holes"
assert n_msk == len(CUBE_IDS), "the mask cache has holes -- common-masking needs it"

# The two caches must cover the SAME cubes, or a paired _cir/_rgb difference is
# computed over a different set on each side and is not a paired difference.
have_rgb = {f.split("__")[0] for f in os.listdir(OUT_RGB) if f.endswith(".npz")}
have_cir = {f.split("__")[0] for f in os.listdir(OUT_CIR) if f.endswith(".npz")}
assert have_rgb == have_cir, (
    f"the two caches cover different cubes: {len(have_rgb - have_cir)} rgb-only, "
    f"{len(have_cir - have_rgb)} cir-only. Every _cir/_rgb paired difference "
    "would silently be computed over a different set on each side.")
print(f"\nboth caches cover the SAME {len(have_rgb)} cubes")

mb_rgb = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_RGB, "*.npz")))
mb_cir = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_CIR, "*.npz")))
mb_msk = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_MSK, "*.npz")))
print(f"cache size  rgb {mb_rgb/1e6:.0f} MB | cir {mb_cir/1e6:.0f} MB | "
      f"masks {mb_msk/1e6:.1f} MB | total {(mb_rgb+mb_cir+mb_msk)/1e6:.0f} MB")


auditing 343 kept cubes

[audit] /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings
[audit]   1715 .npz on disk -> 1715 usable (343 cubes x 5 encoders)
[audit]     dinov2_vitb14            343 cubes
[audit]     imagenet_vit_b16         343 cubes
[audit]     raw_features             343 cubes
[audit]     satlas_s2_swinb_mi_rgb   343 cubes
[audit]     satlas_s2_swinb_rgb      343 cubes
[audit] COMPLETE: all 343 x 5 = 1715 (cube, encoder) pairs present at v3

[audit] /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings_cir
[audit]   1372 .npz on disk -> 1372 usable (343 cubes x 4 encoders)
[audit]     dinov2_vitb14_cir        343 cubes
[audit]     imagenet_vit_b16_cir     343 cubes
[audit]     satlas_s2_swinb_mi_rgb_cir 343 cubes
[audit]     satlas_s2_swinb_rgb_cir  343 cubes
[audit] COMPLETE: all 343 x 4 = 1372 (cube, encoder) pairs present at v3

rgb embeddings 1715 .npz  (expected 343 x 5 = 1715)
cir embeddings 1372 .npz  (expected 343 x 4 = 1372)
ma

## Step 10: THE CROSS-CHECK — same frames, different pixels

This is the cell that makes the phase trustworthy, and it asserts in **two
directions**:

- **Frame selection, timestamps, clear fractions: BIT-IDENTICAL** between the
  two caches. These come from the cube and the mask rule, *before* any network
  sees anything. A single difference means the two caches are not the same
  experiment and no `_cir`-vs-`_rgb` comparison is valid.
- **Embeddings: MATERIALLY DIFFERENT.** If they matched, the band swap never
  reached the stem and the cache is a relabelled copy of the RGB one.

**What is deliberately NOT here.** Phases 1.7 and 1.9 also re-encoded the 20
cubes shared with the published Phase 1.2 cache, as a direct reproducibility
test. **32UQC shares no cube with that cache** — it is a different tile in a
different split — so that check would compare zero pairs and pass vacuously.
It is dropped rather than left in to print a reassuring zero.


In [9]:
from encoders.pipeline import load_encoded

# Drive is a FUSE mount: ~0.5-2 s per .npz open, so 346 x 4 x 2 = 2768 reads is
# well over an hour with no output, which reads as a hang. This invariant is
# STRUCTURAL -- it holds per file or not at all -- so a strided subsample
# spanning the tile is a real check, and n_cubes_checked travels on the CSV.
CHECK_CUBES = 25                      # None = all 346

cubes = sorted(CUBE_IDS)
if CHECK_CUBES:
    cubes = cubes[:: max(1, len(cubes) // CHECK_CUBES)][:CHECK_CUBES]
print(f"cross-check: {len(cubes)} cubes x {len(TIER_A_CIR)} twins = "
      f"{len(cubes)*len(TIER_A_CIR)} pairs, "
      f"{2*len(cubes)*len(TIER_A_CIR)} reads\n", flush=True)

rows, n_same, t0 = [], 0, time.time()
for i, cube in enumerate(cubes, 1):
    stem = os.path.splitext(cube)[0]
    for cir in TIER_A_CIR:
        base = cir[: -len("_cir")]
        a = load_encoded(os.path.join(OUT_RGB, f"{stem}__{base}.npz"))
        b = load_encoded(os.path.join(OUT_CIR, f"{stem}__{cir}.npz"))
        # --- must be IDENTICAL: everything decided before the network ---
        np.testing.assert_array_equal(a.kept_idx, b.kept_idx,
            err_msg=f"{cube}/{base}: frame selection differs between caches")
        np.testing.assert_array_equal(np.asarray(a.timestamps),
                                      np.asarray(b.timestamps),
            err_msg=f"{cube}/{base}: timestamps differ between caches")
        np.testing.assert_allclose(a.clear_frac, b.clear_frac, rtol=0, atol=0)
        # --- must DIFFER: the bands reaching the stem are not the same ---
        d = float(np.abs(a.embeddings - b.embeddings).max())
        same = d == 0.0
        n_same += int(same)
        rows.append({"cube": cube, "encoder": base, "max_abs_pooled_diff": d,
                     "identical": same})
    if i % 5 == 0 or i == len(cubes):
        print(f"  [{i:>3}/{len(cubes)}] {time.time()-t0:5.0f}s", flush=True)

CIR_CHECK = pd.DataFrame(rows)
CIR_CHECK["n_cubes_checked"] = len(cubes)
assert len(CIR_CHECK), "no pair was compared -- the check is vacuous"
print(f"\n{len(CIR_CHECK)} (cube, twin) pairs compared")
print(CIR_CHECK.groupby("encoder")["max_abs_pooled_diff"]
      .agg(["min", "median", "max"]).to_string())
print(f"\nframe selection / timestamps / clear_frac: BIT-IDENTICAL on all "
      f"{len(CIR_CHECK)} pairs")
assert n_same == 0, (
    f"{n_same} pairs have IDENTICAL embeddings. The band swap did not reach "
    "the stem and the cir cache is a relabelled copy of the rgb one.")
print(f"embeddings differ on all {len(CIR_CHECK)} pairs (0 identical)")


cross-check: 25 cubes x 4 twins = 100 pairs, 200 reads

  [  5/25]     1s
  [ 10/25]     2s
  [ 15/25]     2s
  [ 20/25]     3s
  [ 25/25]     4s

100 (cube, twin) pairs compared
                             min     median        max
encoder                                               
dinov2_vitb14           4.121797   5.886057   7.850404
imagenet_vit_b16        3.298298   3.910718   4.827881
satlas_s2_swinb_mi_rgb  1.203047   2.368939   3.500877
satlas_s2_swinb_rgb     5.489067  15.057766  23.781296

frame selection / timestamps / clear_frac: BIT-IDENTICAL on all 100 pairs
embeddings differ on all 100 pairs (0 identical)


## Step 11: The roster, the report, and what to copy down

`cache_roster.csv` is the contract between this notebook and the local fitting
run: it names exactly which cubes the caches cover, so the laptop does not have
to re-derive the roster and cannot silently fit a different one.


In [10]:
ROSTER = pd.DataFrame(
    [{"cube": c, "in_cache": True, "reason": ""} for c in sorted(CUBE_IDS)]
    + [{"cube": c, "in_cache": False, "reason": f"p4_fill_block: {EXCLUDED_REASON}"}
       for c in EXCLUDED_FILL]
    + [{"cube": c, "in_cache": False, "reason": f"encode_failure: {why}"}
       for c, why in dropped])
roster_path = os.path.join(SCALED_ROOT, "cache_roster.csv")
ROSTER.to_csv(roster_path, index=False)

ENCODE_LOG.to_csv(os.path.join(SCALED_ROOT, "phase1_10_extreme_cache.csv"),
                  index=False)
CIR_CHECK.to_csv(os.path.join(SCALED_ROOT, "phase1_10_cir_vs_rgb_check.csv"),
                 index=False)
print(f"wrote {roster_path}  ({int(ROSTER.in_cache.sum())} in cache, "
      f"{int((~ROSTER.in_cache).sum())} excluded)")
print(f"      {SCALED_ROOT}/phase1_10_extreme_cache.csv")
print(f"      {SCALED_ROOT}/phase1_10_cir_vs_rgb_check.csv")

print(textwrap.dedent(f"""
    ------------------------------------------------------------------
    THE EXTREME-TILE CACHE IS READY.

      cubes        {CUBES}
      embeddings   {OUT_RGB}       ({n_rgb} .npz, {mb_rgb/1e6:.0f} MB)
      embeddings_cir {OUT_CIR}     ({n_cir} .npz, {mb_cir/1e6:.0f} MB)
      masks        {OUT_MSK}       ({n_msk} .npz, {mb_msk/1e6:.1f} MB)
      roster       {roster_path}

    COPY DOWN to the laptop, into data/scaled_32UQC/ :

        embeddings/  embeddings_cir/  masks/  cache_roster.csv

    Do NOT copy the cubes -- they are already on the laptop (2.2 GB), and
    data/scaled_32UQC/raw/ is the same 348 files either way.

    THEN, locally, one invocation:

        .venv/bin/python -m scripts.run_p3_extreme --n-jobs 7

    That script re-checks this roster, runs the two-point runtime valve, and
    only then commits to the full grid. Nothing it does needs a GPU and no
    encoder is imported at fit time: it reads these frozen caches.
    ------------------------------------------------------------------
""").strip())


wrote /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/cache_roster.csv  (343 in cache, 5 excluded)
      /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/phase1_10_extreme_cache.csv
      /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/phase1_10_cir_vs_rgb_check.csv
------------------------------------------------------------------
THE EXTREME-TILE CACHE IS READY.

  cubes        /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/raw
  embeddings   /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings       (1715 .npz, 1093 MB)
  embeddings_cir /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/embeddings_cir     (1372 .npz, 1084 MB)
  masks        /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/masks       (343 .npz, 2.1 MB)
  roster       /content/drive/MyDrive/NeurIPS-CCAI-2026/data/scaled_32UQC/cache_roster.csv

COPY DOWN to the laptop, into data/scaled_32UQC/ :

    embeddings/  embeddings_cir/  masks/  cac

## Phase 1.10 is done when

- [ ] Step 3 prints **python 3.10+** and `DEVICE = 'cuda'`. On CPU it still
      works but takes hours; on 3.9 it cannot build DINOv2 at all.
- [ ] Step 4 asserts the roster is **346** cubes from the **`extreme`** split.
- [ ] Step 5 reports **0 failed**.
- [ ] Step 8 reports **0 dropped**. Any dropped cube is out of both caches and
      the masks, and is named in `cache_roster.csv`.
- [ ] Step 9's audits pass: **1730** rgb `.npz`, **1384** cir `.npz`, **346**
      masks, and both caches cover the *same* cubes.
- [ ] **Step 10 passes both directions** — frame selection bit-identical across
      the two caches, embeddings materially different on every pair. Without
      this the cir cache is either a different experiment or a relabelled copy,
      and either way the NIR headline is void.
- [ ] `embeddings/`, `embeddings_cir/`, `masks/` and `cache_roster.csv` copied
      down into `data/scaled_32UQC/`.
